# PathSnap — Dataset Exploration, Model Comparison & Route Decoding

**Reproducible companion notebook for the PathSnap technical report.** This notebook reads **actual saved experiment outputs** generated by the project Python modules; it does not invent or rerun results. It also provides an **optional full training command** for reproduction.

**Data partitions:** 80 `train` trips and 10 `student_test` trips; reserved `evaluator_test` data is **not** used for model selection. **Reference-route alignment produces proxy point labels**, not independently audited GPS ground truth.

**Repository:** https://github.com/manasvidr/PathSnap

**AI use:** ChatGPT was used to assist in preparing this notebook and project documentation; results are sourced from saved project experiment artifacts.

## 1. Setup and experiment files

Open this notebook **from the PathSnap repository root** (or from `notebooks/`). The notebook searches for the project's saved results. Required Python packages: `pandas`, `numpy`, `matplotlib`, `IPython`; training reproduction additionally requires dependencies from the repository.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd()
roots = [cwd, cwd.parent, cwd.parent.parent]
root = next((r for r in roots if (r / 'results' / 'model_comparison.json').exists()), None)
if root is None:
    raise FileNotFoundError('Open this notebook from the PathSnap repository root or notebooks/ folder; results/model_comparison.json is required.')
print('Project root:', root.resolve())

def result_file(folder, name):
    path = root / folder / name
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Copy the actual saved experiment output into the repository.')
    return path

def read_json(folder, name):
    return json.loads(result_file(folder, name).read_text(encoding='utf-8'))

comparison = read_json('results', 'model_comparison.json')
summary = read_json('results', 'summary.json')
print('Selected model:', comparison['selected_model'])
print('Training trips:', comparison['train_trips'], '| Student test trips:', comparison['student_trips'])

## 2. Dataset inventory and data-quality audit

In [ ]:
inventory = pd.read_csv(result_file('results', 'dataset_inventory.csv'))
alignment = pd.read_csv(result_file('results', 'alignment_audit.csv'))
display(inventory.groupby('partition').agg(trips=('trip','count'), observations=('observations','sum'), missing=('missing_or_nonfinite','sum'), bad_coordinates=('bad_coordinates','sum'), timestamp_issues=('out_of_order_timestamps','sum'), gps_gaps_over_5s=('gaps_over_5s','sum')))
display(alignment.groupby('partition')[['within_65m_fraction','candidate_miss_fraction','median_reference_alignment_error_m']].mean().round(4))
fig, ax = plt.subplots(figsize=(9,4))
inventory.groupby('partition')['observations'].sum().plot(kind='bar', ax=ax)
ax.set(title='Observed GPS records by partition', xlabel='Partition', ylabel='GPS observations')
plt.tight_layout(); plt.show()

## 3. Feature engineering and training protocol

Six road-segment candidates are generated per observation using spatial indexing. Each candidate has **13 features**. The training pipeline uses trip-grouped `GroupKFold` cross-validation, preventing observations from the same trip being split across a training/validation fold. The final model is chosen using **training group-CV candidate binary macro-F1**; student-test results are diagnostic only.

In [ ]:
features = comparison['features']
print('Feature count:', len(features))
for i, feature in enumerate(features, 1):
    print(f'{i:2}. {feature}')
print('Training candidate rows:', comparison['train_rows'])
print('Positive candidate labels:', comparison['train_positive'])

## 4. Comparison of five machine-learning models

In [ ]:
records = pd.DataFrame(comparison['records'])
columns = ['model','cv_macro_f1','macro_f1','top1_accuracy','adjacent_direct_continuity','log_loss','inference_seconds','tested_observations']
display(records[columns].sort_values('cv_macro_f1', ascending=False).round(4))
ax = records.set_index('model')[['cv_macro_f1','macro_f1','top1_accuracy']].plot(kind='bar', figsize=(11,5), rot=20)
ax.set(ylabel='Score', ylim=(0,1), title='Five-model comparison (measured results)')
plt.tight_layout(); plt.show()
print('Selected by group-CV:', comparison['selected_model'])

## 5. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, row in zip(axes, comparison['records']):
    cm = np.array(row['confusion_matrix'])
    ax.imshow(cm, cmap='Blues')
    for (i,j), val in np.ndenumerate(cm):
        ax.text(j, i, f'{val:,}', ha='center', va='center', fontsize=8)
    ax.set(title=row['model'], xticks=[0,1], yticks=[0,1], xlabel='Predicted', ylabel='Proxy label')
plt.tight_layout(); plt.show()

## 6. Bayesian optimization (Optuna TPE)

The saved run used **8 trials**, not the script's default 12. For the selected Histogram Gradient Boosting model, the source code specifies: learning rate 0.025–0.18 (log), max leaf nodes 15–39 (step 4), and L2 regularization 0.001–10 (log). Results below come from saved artifacts; tuning is **not** claimed to outperform the baseline.

In [ ]:
tuning = read_json('artifacts','tuning_summary.json')
trials = pd.DataFrame(read_json('artifacts','bayesian_trials.json'))
print('Trials:', tuning['search_trials'])
print('Best parameters:', tuning['selected_params'])
print('Baseline group-CV F1:', round(tuning['baseline_cv_macro_f1'], 6))
print('Best tuned group-CV F1:', round(tuning['best_cv_macro_f1'], 6))
print('Baseline student top-1:', round(tuning['baseline_student']['top1_accuracy'], 6))
print('Tuned student top-1:', round(tuning['tuned_student']['top1_accuracy'], 6))
display(trials[['number','value','parameters','state']])
ax = trials.plot(x='number', y='value', marker='o', legend=False, figsize=(8,4))
ax.set(title='Recorded Bayesian search trials', xlabel='Trial', ylabel='Group-CV macro-F1')
plt.tight_layout(); plt.show()

## 7. HMM/Viterbi route reconstruction

In [ ]:
hmm = read_json('results','hmm_comparison.json')
rows = []
for trip in hmm:
    rows.append({'trip':trip['trip'], 'classifier_accuracy':trip['raw']['accuracy'],
                 'decoded_accuracy':trip['decoded']['accuracy'],
                 'classifier_continuity':trip['raw']['direct_continuity'],
                 'decoded_continuity':trip['decoded']['direct_continuity'],
                 'assessed':trip['decoded']['assessed'], 'decoder_resets':trip['decoder']['resets']})
hmm_df = pd.DataFrame(rows)
display(hmm_df.round(4))
print('Trip-macro mean pointwise accuracy:')
print(hmm_df[['classifier_accuracy','decoded_accuracy','classifier_continuity','decoded_continuity']].mean().round(4))
print('Trips with improved decoded pointwise accuracy:', int((hmm_df.decoded_accuracy > hmm_df.classifier_accuracy).sum()), '/', len(hmm_df))
ax = hmm_df.set_index('trip')[['classifier_accuracy','decoded_accuracy']].plot(kind='bar', figsize=(11,4), rot=35)
ax.set(title='Per-trip pointwise accuracy before and after Viterbi', ylabel='Proxy-label accuracy', ylim=(0,1))
plt.tight_layout(); plt.show()

## 8. Map visualization and evaluator application

The repository's `pathsnap.visuals.folium_trip` generates actual road-network maps. If generated HTML files are present, open them locally. The FastAPI server runs on Laptop 1; Streamlit runs as the evaluator interface. The API predicts from GPS/road graph inputs before separately loading `.route` for scoring.

**Do not use the evaluator partition for training, tuning, or model selection.**

In [ ]:
from IPython.display import HTML
maps = [root/'results/figures/student_trip_comparison.html', root/'results/figures/training_trip_reference.html']
for map_file in maps:
    print(f'{map_file.relative_to(root)}:', 'available' if map_file.exists() else 'not generated in this checkout')
print('API: python -m uvicorn pathsnap.api:app --host 0.0.0.0 --port 8000')
print('UI: python -m streamlit run pathsnap/ui.py --server.address 0.0.0.0 --server.port 8501')

## 9. Optional full experiment reproduction (long-running)

This is **disabled by default** because training takes time and requires the original training/student datasets. Running it invokes the **real** `pathsnap.pipeline` module, regenerates outputs, and may overwrite saved artifacts. The project CLI default is 12 Bayesian trials, but the reported saved run used **8 trials**; this cell explicitly sets 8. The original experiment's sampling settings should be verified against `artifacts/run_config.json` before reproducing.

In [ ]:
RUN_FULL_TRAINING = False  # Change only when original data and dependencies are installed
if RUN_FULL_TRAINING:
    import subprocess, sys
    config = read_json('artifacts', 'run_config.json')
    command = [sys.executable, '-m', 'pathsnap.pipeline',
               '--zip', 'data/student_raw.zip',
               '--root', 'data/raw', '--output', 'results', '--artifacts', 'artifacts',
               '--trials', str(config['bayesian_trials']),
               '--cv-folds', str(config['group_cv_folds']),
               '--candidates', str(config['candidate_k']),
               '--train-observations', str(config['train_observations_per_trip']),
               '--beta', str(config['hmm_beta_m']), '--gap-reset', str(config['gap_reset_seconds'])]
    if config['student_test_observations_per_trip'] is not None:
        command += ['--test-observations', str(config['student_test_observations_per_trip'])]
    subprocess.run(command, cwd=root, check=True)
else:
    print('Full training not run; charts above use real saved experiment outputs.')

## 10. Conclusions and limitations

- Histogram Gradient Boosting was selected by highest training group-CV macro-F1 among five candidates.
- Optuna tuning was executed, but the selected tuned configuration did not beat the baseline CV macro-F1.
- Viterbi improved mean direct route continuity, but decreased mean pointwise accuracy across the ten student trips.
- Pointwise accuracy is computed against **reference-route-aligned proxy labels**, not independently audited ground truth.
- This notebook visualizes existing recorded results; it is **not evidence that notebook cells themselves performed the original training**. Full training is implemented in the repository's Python modules and can be reproduced using the optional cell.

**Submission:** Commit this notebook under `notebooks/PathSnap_Experiment_Notebook.ipynb` and commit the actual `results/` and `artifacts/` JSON/CSV files it reads (subject to repository size limits).